# Extract Wind, Temeprature and Precipitation dimensions
- Use Climate data downloaded via url
- Create clean dimension tables for BI

In [56]:
import pandas as pd
from pathlib import Path
from datetime import datetime

In [57]:
csv_path = Path("../../../../data/clean/Yashaswi/Final_Clean/Atlantic_Climate.csv")
df = pd.read_csv(csv_path, low_memory=False)

In [58]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 949669 entries, 0 to 949668
Data columns (total 17 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Longitude (x)              949669 non-null  float64
 1   Latitude (y)               949669 non-null  float64
 2   Station Name               949669 non-null  str    
 3   Climate ID                 949669 non-null  str    
 4   Date/Time                  949669 non-null  str    
 5   Year                       949669 non-null  int64  
 6   Max Temp (°C)              814288 non-null  float64
 7   Min Temp (°C)              816921 non-null  float64
 8   Mean Temp (°C)             812218 non-null  float64
 9   Heat Deg Days (°C)         812218 non-null  float64
 10  Cool Deg Days (°C)         812218 non-null  float64
 11  Total Rain (mm)            167676 non-null  float64
 12  Total Snow (cm)            163823 non-null  float64
 13  Total Precip (mm)          684101 non-nu

,Longitude (x),Latitude (y),Station Name,Climate ID,Date/Time,Year,Max Temp (°C),Min Temp (°C),Mean Temp (°C),Heat Deg Days (°C),Cool Deg Days (°C),Total Rain (mm),Total Snow (cm),Total Precip (mm),Snow on Grnd (cm),Dir of Max Gust (10s deg),Spd of Max Gust (km/h)
0,-64.51,44.83,AALDERSVILLE,8204020,2021-01-01,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-64.51,44.83,AALDERSVILLE,8204020,2021-01-02,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-64.51,44.83,AALDERSVILLE,8204020,2021-01-03,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-64.51,44.83,AALDERSVILLE,8204020,2021-01-04,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-64.51,44.83,AALDERSVILLE,8204020,2021-01-05,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.columns

Index(['Longitude (x)', 'Latitude (y)', 'Station Name', 'Climate ID',
       'Date/Time', 'Year', 'Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)',
       'Heat Deg Days (°C)', 'Cool Deg Days (°C)', 'Total Rain (mm)',
       'Total Snow (cm)', 'Total Precip (mm)', 'Snow on Grnd (cm)',
       'Dir of Max Gust (10s deg)', 'Spd of Max Gust (km/h)'],
      dtype='str')

## ✅ Create Table with precipitation and snow
Create a new table with only these columns:

- Climate ID
- Date/Time
- Total Rain (mm)
- Total Snow (cm)
- Total Precip (mm)
- Snow on Grnd (cm)
- ✔ Keep the row if any one of these columns has a value
- ❌ Drop the row only if all four are NaN
Columns:

Total Rain (mm)
Total Snow (cm)
Total Precip (mm)
Snow on Grnd (cm)

✅ Create another dimension only for dates from 1 Jan 2010 to 31 Dec 2025 (inclusive) for comparision with AQI

In [5]:
import pandas as pd

df["Date/Time"] = pd.to_datetime(df["Date/Time"])
df_1995_2025 = df[
    (df["Date/Time"] >= "1995-01-01") &
    (df["Date/Time"] <= "2025-12-31")
]
precip_1995_2025 = df_1995_2025[
    [
        "Climate ID",
        "Date/Time",
        "Total Rain (mm)",
        "Total Snow (cm)",
        "Total Precip (mm)",
        "Snow on Grnd (cm)"
    ]
]

In [6]:
precip_1995_2025.info()

<class 'pandas.DataFrame'>
RangeIndex: 949669 entries, 0 to 949668
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Climate ID         949669 non-null  str           
 1   Date/Time          949669 non-null  datetime64[us]
 2   Total Rain (mm)    167676 non-null  float64       
 3   Total Snow (cm)    163823 non-null  float64       
 4   Total Precip (mm)  684101 non-null  float64       
 5   Snow on Grnd (cm)  302071 non-null  float64       
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 49.8 MB


### Drop Nulls

In [7]:
precip_1995_2025_clean = precip_1995_2025.dropna(
    subset=[
        "Total Rain (mm)",
        "Total Snow (cm)",
        "Total Precip (mm)",
        "Snow on Grnd (cm)"
    ],
    how="all"
)

In [8]:
precip_1995_2025_clean.info()

<class 'pandas.DataFrame'>
Index: 698868 entries, 253 to 949668
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Climate ID         698868 non-null  str           
 1   Date/Time          698868 non-null  datetime64[us]
 2   Total Rain (mm)    167676 non-null  float64       
 3   Total Snow (cm)    163823 non-null  float64       
 4   Total Precip (mm)  684101 non-null  float64       
 5   Snow on Grnd (cm)  302071 non-null  float64       
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 42.1 MB


In [9]:
precip_2015_2025 = precip_1995_2025_clean[
    (precip_1995_2025_clean["Date/Time"] >= "2015-01-01") &
    (precip_1995_2025_clean["Date/Time"] <= "2025-12-31")
]

In [10]:

precip_2015_2025["Total Precip (mm)"] = (
    precip_2015_2025["Total Precip (mm)"].round(2)
)

precip_2015_2025.to_parquet(
    r"../../../../data/clean/Yashaswi/Final_Clean/precip_2015_2025.parquet"
)

## Convert daily → monthly
Compute monthly averages for:

Total Rain (mm)
Total Snow (cm)
Total Precip (mm)
Snow on Grnd (cm)


Grouped by Climate ID and Month
Rename column Date/Time → Date
Result should be BI‑ready

## ✅ Step 1: Rename Date/Time → Date
- ✅ Step 2: Ensure Date is datetime (safety check)

In [11]:
precip_1995_2025_clean = precip_1995_2025_clean.rename(
    columns={"Date/Time": "Date"}
)

precip_1995_2025_clean["Date"] = pd.to_datetime(
    precip_1995_2025_clean["Date"]
)

## ✅ Step 3: Create Year‑Month for monthly grouping

In [12]:
precip_1995_2025_clean["Year"]  = precip_1995_2025_clean["Date"].dt.year
precip_1995_2025_clean["Month"] = precip_1995_2025_clean["Date"].dt.month

## ✅ Step 4: Compute monthly averages

In [13]:
precip_monthly = (
    precip_1995_2025_clean
    .groupby(["Climate ID", "Year", "Month"], as_index=False)
    .agg(
        Mean_Total_Rain_mm=("Total Rain (mm)", "mean"),
        Mean_Total_Snow_cm=("Total Snow (cm)", "mean"),
        Mean_Total_Precip_mm=("Total Precip (mm)", "mean"),
        Mean_Snow_on_Grnd_cm=("Snow on Grnd (cm)", "mean"),
        Max_Snow_on_Grnd_cm=("Snow on Grnd (cm)", "max"),
    )
)

## Step 5: Create a proper monthly Date column

In [14]:
precip_monthly["Date"] = pd.to_datetime(
    dict(
        year=precip_monthly["Year"],
        month=precip_monthly["Month"],
        day=1
    )
)

## Step 6: Final column selection & order
- Expected:

Much fewer rows than daily (~30× reduction)
One row per Climate ID per month
Date = first day of each month
Float columns still float64 (OK for now)

In [15]:
precip_monthly = precip_monthly[
    [
        "Climate ID",
        "Date",
        "Mean_Total_Rain_mm",
        "Mean_Total_Snow_cm",
        "Mean_Total_Precip_mm",
        "Mean_Snow_on_Grnd_cm",
        "Max_Snow_on_Grnd_cm",
    ]
]
precip_monthly.head()

,Climate ID,Date,Mean_Total_Rain_mm,Mean_Total_Snow_cm,Mean_Total_Precip_mm,Mean_Snow_on_Grnd_cm,Max_Snow_on_Grnd_cm
0,8100300,1995-01-01,2.129032,2.703226,4.832258,40.000000,61.0
1,8100300,1995-02-01,0.000000,2.628571,2.628571,55.285714,70.0
2,8100300,1995-03-01,0.277419,1.393548,1.670968,67.923077,96.0
3,8100300,1995-04-01,1.453333,0.686667,2.140000,19.333333,42.0
4,8100300,1995-05-01,1.619355,0.083871,1.703226,0.000000,0.0


In [16]:
precip_monthly.info()

<class 'pandas.DataFrame'>
RangeIndex: 24537 entries, 0 to 24536
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Climate ID            24537 non-null  str           
 1   Date                  24537 non-null  datetime64[us]
 2   Mean_Total_Rain_mm    6882 non-null   float64       
 3   Mean_Total_Snow_cm    5820 non-null   float64       
 4   Mean_Total_Precip_mm  24169 non-null  float64       
 5   Mean_Snow_on_Grnd_cm  12754 non-null  float64       
 6   Max_Snow_on_Grnd_cm   12754 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 1.5 MB


## Round & downcast for BI
- ✅ Reduces memory and improves Power BI performance

In [17]:
float_cols = precip_monthly.select_dtypes(include="float").columns
precip_monthly[float_cols] = precip_monthly[float_cols].round(4)
precip_monthly[float_cols] = precip_monthly[float_cols].astype("float32")

In [18]:
precip_monthly.info()

<class 'pandas.DataFrame'>
RangeIndex: 24537 entries, 0 to 24536
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Climate ID            24537 non-null  str           
 1   Date                  24537 non-null  datetime64[us]
 2   Mean_Total_Rain_mm    6882 non-null   float32       
 3   Mean_Total_Snow_cm    5820 non-null   float32       
 4   Mean_Total_Precip_mm  24169 non-null  float32       
 5   Mean_Snow_on_Grnd_cm  12754 non-null  float32       
 6   Max_Snow_on_Grnd_cm   12754 non-null  float32       
dtypes: datetime64[us](1), float32(5), str(1)
memory usage: 1.0 MB


In [19]:
precip_monthly = precip_monthly.rename(
    columns={
        "Mean_Total_Rain_mm": "Monthly_Average_Rain_mm",
        "Mean_Total_Snow_cm": "Monthly_Average_Snow_cm",
        "Mean_Total_Precip_mm": "Monthly_Average_Precip_mm",
        "Mean_Snow_on_Grnd_cm": "Monthly_Snow_on_Grnd_cm",
        "Max_Snow_on_Grnd_cm": "MonthlyMax_Snow_on_Grnd_cm",
    }
)
precip_monthly.head()

,Climate ID,Date,Monthly_Average_Rain_mm,Monthly_Average_Snow_cm,Monthly_Average_Precip_mm,Monthly_Snow_on_Grnd_cm,MonthlyMax_Snow_on_Grnd_cm
0,8100300,1995-01-01,2.1290,2.7032,4.8323,40.000000,61.0
1,8100300,1995-02-01,0.0000,2.6286,2.6286,55.285702,70.0
2,8100300,1995-03-01,0.2774,1.3935,1.6710,67.923103,96.0
3,8100300,1995-04-01,1.4533,0.6867,2.1400,19.333300,42.0
4,8100300,1995-05-01,1.6194,0.0839,1.7032,0.000000,0.0


### Extract Precipitation data to csv

In [20]:
precip_monthly.to_csv(
    r"../../../../data/clean/Yashaswi/Final_Clean/precip_monthly.csv",
    index=False
)

## Create the three new DataFrames with the specified names

In [21]:
df_wind = df[
    [
        'Climate ID',
        'Date/Time',
        'Dir of Max Gust (10s deg)',
        'Spd of Max Gust (km/h)'
    ]
]

df_station = df[
    [
        'Longitude (x)',
        'Latitude (y)',
        'Station Name',
        'Climate ID'
    ]
]
df_temp = df[
    [
        'Climate ID',
        'Date/Time',
        'Max Temp (°C)',
        'Min Temp (°C)',
        'Mean Temp (°C)',
        'Heat Deg Days (°C)',
        'Cool Deg Days (°C)'
    ]
]

In [22]:
df_temp.info()

<class 'pandas.DataFrame'>
RangeIndex: 949669 entries, 0 to 949668
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Climate ID          949669 non-null  str           
 1   Date/Time           949669 non-null  datetime64[us]
 2   Max Temp (°C)       814288 non-null  float64       
 3   Min Temp (°C)       816921 non-null  float64       
 4   Mean Temp (°C)      812218 non-null  float64       
 5   Heat Deg Days (°C)  812218 non-null  float64       
 6   Cool Deg Days (°C)  812218 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 57.1 MB


##  Make df_station contain only unique Climate IDs (EDA for dim_Station)
- To make df_station contain only unique Climate IDs (one row per station), drop duplicate rows based on the Climate ID column.

In [23]:
df_station.info()
df_station = df_station.drop_duplicates(subset='Climate ID').reset_index(drop=True)

<class 'pandas.DataFrame'>
RangeIndex: 949669 entries, 0 to 949668
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Longitude (x)  949669 non-null  float64
 1   Latitude (y)   949669 non-null  float64
 2   Station Name   949669 non-null  str    
 3   Climate ID     949669 non-null  str    
dtypes: float64(2), str(2)
memory usage: 47.9 MB


In [28]:
df_station.value_counts()

Longitude (x)  Latitude (y)  Station Name         Climate ID
-64.51         44.83         AALDERSVILLE         8204020       1
-54.00         47.31         ARGENTIA (AUT)       8400104       1
-67.72         46.71         AROOSTOOK            8100300       1
-65.47         43.45         BACCARO PT           8200255       1
-56.07         48.97         BADGER (AUT)         8400301       1
                                                               ..
-67.54         46.15         WOODSTOCK NEWBRIDGE  8105603       1
-59.31         47.71         WRECKHOUSE           8404343       1
-66.09         43.83         YARMOUTH A           8206495       1
                                                  8206496       1
                             YARMOUTH RCS         8206491       1
Name: count, Length: 136, dtype: int64

## EDA for dim_Station 
- compare your station list (df_station) with df_atlantic and then export the cleaned/compared station list to CSV.

In [27]:
df_stationlist.info()

NameError: name 'df_stationlist' is not defined

**Identify Climate IDs NOT matching**
  - Climate IDs in df_station but missing from df_stationlist
  - Climate IDs in df_stationlist but missing from df_station
  - Quick counts (recommended sanity check)

In [26]:
not_in_clean = df_station[
    ~df_station['Climate ID'].isin(df_stationlist['Climate ID'])
]
not_in_station = df_stationlist[
    ~df_stationlist['Climate ID'].isin(df_station['Climate ID'])
]
print("In df_station but not in df_stationlist:",
      not_in_clean['Climate ID'].nunique())

print("In df_stationlist but not in df_station:",
      not_in_station['Climate ID'].nunique())

NameError: name 'df_stationlist' is not defined

**1. Identify the mismatching Climate ID**

In [29]:
missing_ids = df_stationlist.loc[
    ~df_stationlist['Climate ID'].isin(df_station['Climate ID']),
    'Climate ID'
]

missing_ids

69    8502588
Name: Climate ID, dtype: str

**2. Check if this Climate ID exists in temperature data**

In [30]:
missing_climate_id = missing_ids.iloc[0]
df_temp_missing = df_temp[df_temp['Climate ID'] == missing_climate_id]

**3. Inspect the temperature values**

In [31]:
df_temp_missing.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Climate ID          0 non-null      str           
 1   Date/Time           0 non-null      datetime64[us]
 2   Max Temp (°C)       0 non-null      float64       
 3   Min Temp (°C)       0 non-null      float64       
 4   Mean Temp (°C)      0 non-null      float64       
 5   Heat Deg Days (°C)  0 non-null      float64       
 6   Cool Deg Days (°C)  0 non-null      float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 132.0 bytes


**Look for it in df_stationlist**

In [32]:
df_stationlist_info = df_stationlist[
    df_stationlist['Climate ID'] == missing_climate_id
]

df_stationlist_info

,Station Name,Province,Latitude,Longitude,Climate ID,First Year,Last Year
69,MAKKOVIK A,NEWFOUNDLAND AND LABRADOR,55.08,-59.19,8502588,2015,2026


In [33]:
df_station.info()

<class 'pandas.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Longitude (x)  136 non-null    float64
 1   Latitude (y)   136 non-null    float64
 2   Station Name   136 non-null    str    
 3   Climate ID     136 non-null    str    
dtypes: float64(2), str(2)
memory usage: 7.2 KB


In [34]:
df_stationlist.info()

<class 'pandas.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Station Name  137 non-null    str    
 1   Province      137 non-null    str    
 2   Latitude      137 non-null    float64
 3   Longitude     137 non-null    float64
 4   Climate ID    137 non-null    str    
 5   First Year    137 non-null    int64  
 6   Last Year     137 non-null    int64  
dtypes: float64(2), int64(2), str(3)
memory usage: 12.8 KB


# Finalize dim_station_clean

In [35]:
station_province_lookup = (
    df_stationlist[["Climate ID", "Province"]]
    .drop_duplicates(subset=["Climate ID"])
)
df_station = df_station.merge(
    station_province_lookup,
    on="Climate ID",
    how="left"
)

In [36]:
df_station.info()

<class 'pandas.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Longitude (x)  136 non-null    float64
 1   Latitude (y)   136 non-null    float64
 2   Station Name   136 non-null    str    
 3   Climate ID     136 non-null    str    
 4   Province       136 non-null    str    
dtypes: float64(2), str(3)
memory usage: 10.6 KB


In [38]:
df_station.to_csv(
    r"../../../../data/clean/Yashaswi/Final_Clean/dim_station_clean.csv",
    index=False
)

✅ Final conclusion (important)
This Climate ID:

exists only in Stationlist_clean.csv
has no station metadata in my main dataset
has no temperature records

✅ Therefore, it is invalid for analysis
✅ It should be removed from the station list

## Temperature dim table EDA

In [59]:
df_temp.info()

<class 'pandas.DataFrame'>
RangeIndex: 949669 entries, 0 to 949668
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Climate ID          949669 non-null  str           
 1   Date/Time           949669 non-null  datetime64[us]
 2   Max Temp (°C)       814288 non-null  float64       
 3   Min Temp (°C)       816921 non-null  float64       
 4   Mean Temp (°C)      812218 non-null  float64       
 5   Heat Deg Days (°C)  812218 non-null  float64       
 6   Cool Deg Days (°C)  812218 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 57.1 MB


## Count NULL (NaN) values per column

In [60]:
temp_cols = [
    'Max Temp (°C)',
    'Min Temp (°C)',
    'Mean Temp (°C)',
    'Heat Deg Days (°C)',
    'Cool Deg Days (°C)'
]
null_counts = df_temp[temp_cols].isna().sum()
null_counts

Max Temp (°C)         135381
Min Temp (°C)         132748
Mean Temp (°C)        137451
Heat Deg Days (°C)    137451
Cool Deg Days (°C)    137451
dtype: int64

## 1. Find rows where ALL temperature columns are NULL

In [61]:
all_null_rows = df_temp[temp_cols].isna().all(axis=1)
all_null_count = all_null_rows.sum()
all_null_count

np.int64(130676)

## 2. Inspect those rows

In [62]:
df_temp_all_null = df_temp[all_null_rows]
df_temp_all_null.head()
df_temp_all_null.info()

<class 'pandas.DataFrame'>
Index: 130676 entries, 0 to 949613
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Climate ID          130676 non-null  str           
 1   Date/Time           130676 non-null  datetime64[us]
 2   Max Temp (°C)       0 non-null       float64       
 3   Min Temp (°C)       0 non-null       float64       
 4   Mean Temp (°C)      0 non-null       float64       
 5   Heat Deg Days (°C)  0 non-null       float64       
 6   Cool Deg Days (°C)  0 non-null       float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 8.9 MB


## Cleanup "all null" rows

In [63]:
df_temp_clean = df_temp[~all_null_rows].reset_index(drop=True)
df_temp_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 818993 entries, 0 to 818992
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Climate ID          818993 non-null  str           
 1   Date/Time           818993 non-null  datetime64[us]
 2   Max Temp (°C)       814288 non-null  float64       
 3   Min Temp (°C)       816921 non-null  float64       
 4   Mean Temp (°C)      812218 non-null  float64       
 5   Heat Deg Days (°C)  812218 non-null  float64       
 6   Cool Deg Days (°C)  812218 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 49.3 MB


**Current situation

df_temp_all_null → ✅ rows where all temp columns were null (already isolated)
df_temp_clean → ✅ all‑null rows removed
Now inside df_temp_clean, there are:

✅ fully complete rows and Partially null rows**

## Find rows with any nulls (after all‑null rows are removed)
 -  Count partially‑null rows

In [64]:
partial_null_rows = df_temp_clean[temp_cols].isna().any(axis=1)
partial_null_count = partial_null_rows.sum()
partial_null_count

np.int64(6779)

In [65]:
df_temp_partial_null = df_temp_clean[partial_null_rows]

df_temp_partial_null.head()

,Climate ID,Date/Time,Max Temp (°C),Min Temp (°C),Mean Temp (°C),Heat Deg Days (°C),Cool Deg Days (°C)
1564,8400104,1995-01-23,NaN,-0.9,NaN,NaN,NaN
1571,8400104,1995-01-30,NaN,-6.2,NaN,NaN,NaN
1573,8400104,1995-02-02,NaN,-5.5,NaN,NaN,NaN
1579,8400104,1995-02-08,NaN,-7.9,NaN,NaN,NaN
1584,8400104,1995-02-13,-4.8,NaN,NaN,NaN,NaN


In [66]:
df_temp_partial_null.info()

<class 'pandas.DataFrame'>
Index: 6779 entries, 1564 to 816236
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Climate ID          6779 non-null   str           
 1   Date/Time           6779 non-null   datetime64[us]
 2   Max Temp (°C)       2074 non-null   float64       
 3   Min Temp (°C)       4707 non-null   float64       
 4   Mean Temp (°C)      4 non-null      float64       
 5   Heat Deg Days (°C)  4 non-null      float64       
 6   Cool Deg Days (°C)  4 non-null      float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 470.9 KB


# NEW RULE DEFINITION FOR ROW VALIDITY 
Remove rows that match this specific invalid pattern:

- ❌ Max Temp, Min Temp, and Mean Temp are ALL null
- ✅ Heat Deg Days and Cool Deg Days HAVE values

This is a very reasonable quality rule for climate data.

In [67]:
temp_cols_core = [
    'Max Temp (°C)',
    'Min Temp (°C)',
    'Mean Temp (°C)'
]

degday_cols = [
    'Heat Deg Days (°C)',
    'Cool Deg Days (°C)'
]
invalid_rows = (
    df_temp_clean[temp_cols_core].isna().all(axis=1) &      # all temps missing
    df_temp_clean[degday_cols].notna().any(axis=1)          # but deg days present
)
invalid_rows.sum()

np.int64(0)

# NEW FILLING RULE FOR ROW VALIDITY
✅ Filling rules (restated clearly)
For rows where Mean Temp (°C) is NA:


✅ If BOTH Max and Min are present
→ Mean = (Max + Min) / 2


✅ If ONLY ONE of Max or Min is present
→ Mean = the available value


❌ If BOTH Max and Min are missing
→ Leave Mean as NA (I have already removed fully-null rows)

In [68]:
df_t = df_temp_clean.copy()
mask_both = (
    df_t['Mean Temp (°C)'].isna() &
    df_t['Max Temp (°C)'].notna() &
    df_t['Min Temp (°C)'].notna()
)

df_t.loc[mask_both, 'Mean Temp (°C)'] = (
    df_t.loc[mask_both, ['Max Temp (°C)', 'Min Temp (°C)']].mean(axis=1)
)
mask_only_max = (
    df_t['Mean Temp (°C)'].isna() &
    df_t['Max Temp (°C)'].notna() &
    df_t['Min Temp (°C)'].isna()
)

df_t.loc[mask_only_max, 'Mean Temp (°C)'] = df_t.loc[
    mask_only_max, 'Max Temp (°C)'
]
mask_only_min = (
    df_t['Mean Temp (°C)'].isna() &
    df_t['Min Temp (°C)'].notna() &
    df_t['Max Temp (°C)'].isna()
)

df_t.loc[mask_only_min, 'Mean Temp (°C)'] = df_t.loc[
    mask_only_min, 'Min Temp (°C)'
]
df_t['Mean Temp (°C)'].isna().sum()

np.int64(0)

# Spot‑check correctness

In [69]:
df_t.loc[
    mask_both | mask_only_max | mask_only_min,
    ['Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)']
].head(10)

,Max Temp (°C),Min Temp (°C),Mean Temp (°C)
1564,NaN,-0.9,-0.9
1571,NaN,-6.2,-6.2
1573,NaN,-5.5,-5.5
1579,NaN,-7.9,-7.9
1584,-4.8,NaN,-4.8
1594,NaN,-7.5,-7.5
1596,NaN,-0.2,-0.2
1607,NaN,0.9,0.9
1612,NaN,-6.6,-6.6
1614,NaN,-0.5,-0.5


# Finalize dim_temp data

In [70]:
df_temp_final = df_t
df_temp_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 818993 entries, 0 to 818992
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Climate ID          818993 non-null  str           
 1   Date/Time           818993 non-null  datetime64[us]
 2   Max Temp (°C)       814288 non-null  float64       
 3   Min Temp (°C)       816921 non-null  float64       
 4   Mean Temp (°C)      818993 non-null  float64       
 5   Heat Deg Days (°C)  812218 non-null  float64       
 6   Cool Deg Days (°C)  812218 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 49.3 MB


In [71]:
degday_cols = [
    'Heat Deg Days (°C)',
    'Cool Deg Days (°C)'
]
null_counts = df_temp_final[degday_cols].isna().sum()
null_counts

Heat Deg Days (°C)    6775
Cool Deg Days (°C)    6775
dtype: int64

In [72]:
zero_counts = (df_temp_final[degday_cols] == 0).sum()
zero_counts

Heat Deg Days (°C)     87776
Cool Deg Days (°C)    727545
dtype: int64

In [73]:
(
    (df_temp_final['Heat Deg Days (°C)'] > 0) &
    (df_temp_final['Cool Deg Days (°C)'] > 0)
).sum()


np.int64(0)

In [74]:
df_temp_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 818993 entries, 0 to 818992
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Climate ID          818993 non-null  str           
 1   Date/Time           818993 non-null  datetime64[us]
 2   Max Temp (°C)       814288 non-null  float64       
 3   Min Temp (°C)       816921 non-null  float64       
 4   Mean Temp (°C)      818993 non-null  float64       
 5   Heat Deg Days (°C)  812218 non-null  float64       
 6   Cool Deg Days (°C)  812218 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 49.3 MB


In [75]:
df_temp_final = df_temp_final.rename(columns={"Date/Time": "Date"})
df_temp_final = df_temp_final[
   (df_temp_final["Date"] <= "2025-12-31")
]
df_temp_final.head()

,Climate ID,Date,Max Temp (°C),Min Temp (°C),Mean Temp (°C),Heat Deg Days (°C),Cool Deg Days (°C)
0,8204020,2021-09-11,18.7,8.7,13.7,4.3,0.0
1,8204020,2021-09-12,22.4,9.8,16.1,1.9,0.0
2,8204020,2021-09-13,20.8,11.0,15.9,2.1,0.0
3,8204020,2021-09-14,17.2,6.9,12.1,5.9,0.0
4,8204020,2021-09-15,20.2,5.7,13.0,5.0,0.0


In [76]:
df_temp_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 818993 entries, 0 to 818992
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Climate ID          818993 non-null  str           
 1   Date                818993 non-null  datetime64[us]
 2   Max Temp (°C)       814288 non-null  float64       
 3   Min Temp (°C)       816921 non-null  float64       
 4   Mean Temp (°C)      818993 non-null  float64       
 5   Heat Deg Days (°C)  812218 non-null  float64       
 6   Cool Deg Days (°C)  812218 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 49.3 MB


## Changing data point for one station for 20 April 2008, as max temp it is not right
- Assumption ( Station ID did not record max temperature correctly)

In [77]:
err = df_temp_final.loc[
    (df_temp_final["Climate ID"] == "8204453") &
    (df_temp_final["Date"].dt.year == 2008) &
    (df_temp_final["Date"].dt.month == 4), "Max Temp (°C)"
]
err

561805    11.0
561806    10.5
561807    16.0
561808    14.5
561809    45.0
561810     7.0
561811     8.0
561812    15.0
Name: Max Temp (°C), dtype: float64

In [78]:
df_temp_final.loc[561809]

Climate ID                        8204453
Date                  2008-04-20 00:00:00
Max Temp (°C)                        45.0
Min Temp (°C)                        -2.0
Mean Temp (°C)                       21.5
Heat Deg Days (°C)                    0.0
Cool Deg Days (°C)                    3.5
Name: 561809, dtype: object

## run below 2 cells only once for replacing error max temperature with average max temperature of the month

In [81]:
df_temp_final.loc[
    (df_temp_final["Climate ID"] == "8204453") &
    (df_temp_final["Date"].dt.year == 2008) &
    (df_temp_final["Date"].dt.month == 4),
    ["Date", "Max Temp (°C)"]
].sort_values("Date")

,Date,Max Temp (°C)
561805,2008-04-07,11.000
561806,2008-04-11,10.500
561807,2008-04-17,16.000
561808,2008-04-18,14.500
561809,2008-04-20,15.875
561810,2008-04-25,7.000
561811,2008-04-26,8.000
561812,2008-04-27,15.000


In [82]:
df_temp_final.to_parquet( r"../../../../data/clean/Yashaswi/Final_Clean/temp_clean_Daily.parquet")

In [83]:
df_temp_final.head()

,Climate ID,Date,Max Temp (°C),Min Temp (°C),Mean Temp (°C),Heat Deg Days (°C),Cool Deg Days (°C)
0,8204020,2021-09-11,18.7,8.7,13.7,4.3,0.0
1,8204020,2021-09-12,22.4,9.8,16.1,1.9,0.0
2,8204020,2021-09-13,20.8,11.0,15.9,2.1,0.0
3,8204020,2021-09-14,17.2,6.9,12.1,5.9,0.0
4,8204020,2021-09-15,20.2,5.7,13.0,5.0,0.0


In [84]:
temp_2015_2025 = df_temp_final[
    (df_temp_final["Date"] >= "2015-01-01") &
    (df_temp_final["Date"] <= "2025-12-31")
]

In [85]:
temp_2015_2025["Mean Temp (°C)"] = temp_2015_2025["Mean Temp (°C)"].round(2)

temp_2015_2025.to_parquet( r"../../../../data/clean/Yashaswi/Final_Clean/temp_2015.parquet")

## Roll daily → monthly temperature data with monthly averages, grouped by:

Climate ID
Year + Month

And produce a BI‑ready monthly temperature table.

## ✅ Step 1: Ensure Date is datetime (safe check)

In [86]:
df_temp_final["Date"] = pd.to_datetime(df_temp_final["Date"])

## ✅ Step 2: Create Year and Month columns

In [87]:
df_temp_final["Year"] = df_temp_final["Date"].dt.year
df_temp_final["Month"] = df_temp_final["Date"].dt.month

## ✅ Step 3: Monthly aggregation (AVERAGES)

In [88]:

temp_monthly = (
    df_temp_final
    .groupby(["Climate ID", "Year", "Month"], as_index=False)
    .agg(
        Monthly_Average_Max_Temp_C=("Max Temp (°C)", "mean"),
        Monthly_Average_Min_Temp_C=("Min Temp (°C)", "mean"),
        Monthly_Average_Mean_Temp_C=("Mean Temp (°C)", "mean"),
        Monthly_Max_Temp_C=("Max Temp (°C)", "max"),  
        Monthly_Min_Temp_C=("Min Temp (°C)", "min"),   
        Monthly_Average_Heat_Deg_Days_C=("Heat Deg Days (°C)", "mean"),
        Monthly_Average_Cool_Deg_Days_C=("Cool Deg Days (°C)", "mean"),
    )
)


## ✅ Create monthly Date column (BI‑friendly)

In [89]:
temp_monthly["Date"] = pd.to_datetime(
    dict(
        year=temp_monthly["Year"],
        month=temp_monthly["Month"],
        day=1
    )
)

## ✅ Final column order

In [90]:
temp_monthly = temp_monthly[
    [
        "Climate ID",
        "Date",
        "Monthly_Average_Max_Temp_C",
        "Monthly_Average_Min_Temp_C",
        "Monthly_Average_Mean_Temp_C",
        "Monthly_Max_Temp_C",
        "Monthly_Min_Temp_C",
        "Monthly_Average_Heat_Deg_Days_C",
        "Monthly_Average_Cool_Deg_Days_C",
    ]
]
 # Round & optimize for BI

float_cols = temp_monthly.select_dtypes(include="float").columns
temp_monthly[float_cols] = temp_monthly[float_cols].round(4)
temp_monthly[float_cols] = temp_monthly[float_cols].astype("float32")

In [91]:
temp_monthly.head()

,Climate ID,Date,Monthly_Average_Max_Temp_C,Monthly_Average_Min_Temp_C,Monthly_Average_Mean_Temp_C,Monthly_Max_Temp_C,Monthly_Min_Temp_C,Monthly_Average_Heat_Deg_Days_C,Monthly_Average_Cool_Deg_Days_C
0,8100300,1995-01-01,-3.8548,-14.2581,-9.0613,11.5,-40.0,27.061300,0.0
1,8100300,1995-02-01,-6.1964,-20.9464,-13.6000,5.0,-32.0,31.600000,0.0
2,8100300,1995-03-01,1.6613,-8.2097,-3.2871,10.5,-26.0,21.287100,0.0
3,8100300,1995-04-01,6.5667,-4.6667,0.9600,14.0,-15.5,17.040001,0.0
4,8100300,1995-05-01,17.5161,3.4032,10.4839,24.5,-3.0,7.516100,0.0


In [92]:
temp_monthly.info()

<class 'pandas.DataFrame'>
RangeIndex: 28105 entries, 0 to 28104
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Climate ID                       28105 non-null  str           
 1   Date                             28105 non-null  datetime64[us]
 2   Monthly_Average_Max_Temp_C       28086 non-null  float32       
 3   Monthly_Average_Min_Temp_C       28088 non-null  float32       
 4   Monthly_Average_Mean_Temp_C      28105 non-null  float32       
 5   Monthly_Max_Temp_C               28086 non-null  float32       
 6   Monthly_Min_Temp_C               28088 non-null  float32       
 7   Monthly_Average_Heat_Deg_Days_C  28070 non-null  float32       
 8   Monthly_Average_Cool_Deg_Days_C  28070 non-null  float32       
dtypes: datetime64[us](1), float32(7), str(1)
memory usage: 1.4 MB


In [93]:
temp_monthly["Date"].min(), temp_monthly["Date"].max()

(Timestamp('1995-01-01 00:00:00'), Timestamp('2025-12-01 00:00:00'))

In [94]:
temp_monthly.to_csv(
    r"../../../../data/clean/Yashaswi/Final_Clean/monthly_temperature_1995_2025.csv",
    index=False
)

## Create csv for daily data : 2010 to 2025

# EDA for dim_wind

In [77]:
df_wind.info()

<class 'pandas.DataFrame'>
RangeIndex: 949669 entries, 0 to 949668
Data columns (total 4 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Climate ID                 949669 non-null  str           
 1   Date/Time                  949669 non-null  datetime64[us]
 2   Dir of Max Gust (10s deg)  453421 non-null  float64       
 3   Spd of Max Gust (km/h)     458611 non-null  float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 35.3 MB


## Null check

In [78]:

wind_cols = [
    'Dir of Max Gust (10s deg)',
    'Spd of Max Gust (km/h)'
]

df_wind[wind_cols].isna().sum()

Dir of Max Gust (10s deg)    496248
Spd of Max Gust (km/h)       491058
dtype: int64

In [79]:

any_null_rows = df_wind[wind_cols].isna().any(axis=1)
any_null_rows.sum()


np.int64(496248)

In [80]:

both_null_rows = df_wind[wind_cols].isna().all(axis=1)
both_null_rows.sum()


np.int64(491058)

# ✅ How to interpret this (important)


✅ Dir or Speed = NULL usually means:

No gust observed that day
Sensor limitation
Reporting rules (gusts often only logged when above threshold)

✅ Both NULL is very common in climatological wind datasets
→ not an error

⚠️ Should not drop these automatically unless your analysis requires gust data.

## For wind gust data: remove nulls logic

- ✅ REMOVE rows where both
Dir of Max Gust (10s deg) AND Spd of Max Gust (km/h) are NaN
- ✅ KEEP rows where at least one of them is present

This matches standard climatological practice.

In [81]:
df_wind_clean = df_wind[~both_null_rows].reset_index(drop=True)
df_wind_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 458611 entries, 0 to 458610
Data columns (total 4 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Climate ID                 458611 non-null  str           
 1   Date/Time                  458611 non-null  datetime64[us]
 2   Dir of Max Gust (10s deg)  453421 non-null  float64       
 3   Spd of Max Gust (km/h)     458611 non-null  float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 17.1 MB


In [82]:
df_wind_clean = df_wind_clean.rename(columns={"Date/Time": "Date"})
df_wind_clean.head()

,Climate ID,Date,Dir of Max Gust (10s deg),Spd of Max Gust (km/h)
0,8204020,2021-09-11,28.0,43.0
1,8204020,2021-09-12,22.0,49.0
2,8204020,2021-09-13,31.0,34.0
3,8204020,2021-09-15,18.0,43.0
4,8204020,2021-09-16,23.0,33.0


In [83]:
df_wind_clean[
    ['Dir of Max Gust (10s deg)', 'Spd of Max Gust (km/h)']
].isna().all(axis=1).sum()


np.int64(0)

### dim_wind - Columns after cleaning

| Column                         | Non‑null count | Interpretation                                      |
|--------------------------------|----------------|-----------------------------------------------------|
| Climate ID                     | 458,611        | ✅ Intact                                            |
| Date/Time                      | 458,611        | ✅ Intact                                            |
| Dir of Max Gust (10s deg)      | 453,421        | Some gusts lack direction (normal)                  |
| Spd of Max Gust (km/h)         | 458,611        | Every remaining row has gust speed                  |

In [84]:
# Convert Date column to datetime
df_wind_clean["Date"] = pd.to_datetime(df_wind_clean["Date"])

# Filter date range
df_wind_clean = df_wind_clean[
    (df_wind_clean["Date"] >= "1995-01-01") &
    (df_wind_clean["Date"] <= "2025-12-31")
]

In [85]:
df_wind_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 458611 entries, 0 to 458610
Data columns (total 4 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Climate ID                 458611 non-null  str           
 1   Date                       458611 non-null  datetime64[us]
 2   Dir of Max Gust (10s deg)  453421 non-null  float64       
 3   Spd of Max Gust (km/h)     458611 non-null  float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 17.1 MB


In [86]:
df_wind_clean.head()

,Climate ID,Date,Dir of Max Gust (10s deg),Spd of Max Gust (km/h)
0,8204020,2021-09-11,28.0,43.0
1,8204020,2021-09-12,22.0,49.0
2,8204020,2021-09-13,31.0,34.0
3,8204020,2021-09-15,18.0,43.0
4,8204020,2021-09-16,23.0,33.0


In [87]:
df_wind_clean[
    ["Dir of Max Gust (10s deg)", "Spd of Max Gust (km/h)"]
].isna().any()

Dir of Max Gust (10s deg)     True
Spd of Max Gust (km/h)       False
dtype: bool

In [88]:
df_wind_clean = df_wind_clean.sort_values(by="Date", ascending=True)

In [89]:
df_wind_clean["wind_key"] = range(100, 100 + len(df_wind_clean))
cols = ["wind_key"] + [c for c in df_wind_clean.columns if c != "wind_key"]
df_wind_clean = df_wind_clean[cols]
df_wind_clean.reset_index(drop=True)

,wind_key,Climate ID,Date,Dir of Max Gust (10s deg),Spd of Max Gust (km/h)
0,100,8205090,1995-01-02,30.0,50.0
1,101,8202000,1995-01-03,29.0,43.0
2,102,8402520,1995-01-03,28.0,43.0
3,103,8205090,1995-01-03,31.0,59.0
4,104,8501900,1995-01-04,25.0,44.0
...,...,...,...,...,...
458606,458706,8101605,2025-12-31,26.0,34.0
458607,458707,8401565,2025-12-31,21.0,84.0
458608,458708,8201780,2025-12-31,21.0,55.0
458609,458709,840B053,2025-12-31,20.0,51.0


In [90]:
df_wind_clean.info()

<class 'pandas.DataFrame'>
Index: 458611 entries, 331980 to 458610
Data columns (total 5 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   wind_key                   458611 non-null  int64         
 1   Climate ID                 458611 non-null  str           
 2   Date                       458611 non-null  datetime64[us]
 3   Dir of Max Gust (10s deg)  453421 non-null  float64       
 4   Spd of Max Gust (km/h)     458611 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(1)
memory usage: 24.1 MB


## Export Daily wind data to csv

In [92]:
df_wind_clean.to_parquet(
    r"../../../../data/clean/Yashaswi/Final_Clean/df_wind_clean.parquet",
    index=False
)

In [93]:
wind_2015_2025 = df_wind_clean[
    (df_wind_clean["Date"] >= "2015-01-01") &
    (df_wind_clean["Date"] <= "2025-12-31")
]

In [95]:
wind_2015_2025.to_parquet(
    r"../../../../data/clean/Yashaswi/Final_Clean/wind_2015_2025.parquet",
    index=False
)

In [96]:
wind_2015_2025.head()

,wind_key,Climate ID,Date,Dir of Max Gust (10s deg),Spd of Max Gust (km/h)
234517,166313,840KN90,2015-01-01,21.0,44.0
63000,166314,8200774,2015-01-01,18.0,33.0
422107,166315,8404025,2015-01-01,26.0,56.0
147343,166316,8401703,2015-01-01,22.0,35.0
455676,166317,8206491,2015-01-01,26.0,48.0


## ✅ 1️⃣ Create DAILY wind CSV (2010–2025)

## ✅ Step 1: Filter dates

In [97]:
df_wind_daily_2010_2025 = df_wind_clean[
    (df_wind_clean["Date"] >= "2010-01-01") &
    (df_wind_clean["Date"] <= "2025-12-31")
].copy()

## Step 2:Drop surrogate key for BI
The wind_key is not useful after aggregation.

In [98]:
df_wind_daily_2010_2025 = df_wind_daily_2010_2025.drop(columns=["wind_key"])

In [99]:
df_wind_daily_2010_2025.info()

<class 'pandas.DataFrame'>
Index: 377541 entries, 292818 to 458610
Data columns (total 4 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Climate ID                 377541 non-null  str           
 1   Date                       377541 non-null  datetime64[us]
 2   Dir of Max Gust (10s deg)  373415 non-null  float64       
 3   Spd of Max Gust (km/h)     377541 non-null  float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 17.0 MB


## ✅ Step 3: Export daily CSV

## ✅ 2️⃣ Create MONTHLY rolled‑up wind table
- 🔬 Aggregation logic (important)
Wind gust data is event‑based, so we should aggregate as:

- Variable                  |    Monthly aggregation
- Spd of Max Gust (km/h)    |    MAX (strongest gust in month)
- Dir of Max Gust (10s deg) |    Mean (prevailing gust direction)

This is standard climatological practice.

## ✅ Step 1: Add Year & Month

In [100]:
df_wind_clean["Year"] = df_wind_clean["Date"].dt.year
df_wind_clean["Month"] = df_wind_clean["Date"].dt.month
df_wind_clean["Date"] = pd.to_datetime(df_wind_clean["Date"])

## ✅ Step 2: Monthly aggregation

In [101]:

df_wind_1995_2025 = df_wind_clean[
    (df_wind_clean["Date"] >= "1995-01-01") &
    (df_wind_clean["Date"] <= "2025-12-31")
].copy()

df_wind_1995_2025["Year"] = df_wind_1995_2025["Date"].dt.year
df_wind_1995_2025["Month"] = df_wind_1995_2025["Date"].dt.month

wind_monthly_1995_2025 = (
    df_wind_1995_2025
    .groupby(["Climate ID", "Year", "Month"], as_index=False)
    .agg(
        Monthly_Max_Gust_kmh=("Spd of Max Gust (km/h)", "max"),
        Monthly_Mean_Gust_Direction_10sdeg=("Dir of Max Gust (10s deg)", "mean"),
    )
)

## ✅ Step 3: Create monthly Date column (first of month)

In [102]:
wind_monthly_1995_2025["Date"] = pd.to_datetime(
    dict(
        year=wind_monthly_1995_2025["Year"],
        month=wind_monthly_1995_2025["Month"],
        day=1
    )
)

## ✅ Step 4: Final column order

In [103]:
wind_monthly_1995_2025 = wind_monthly_1995_2025[
    [
        "Climate ID",
        "Date",
        "Monthly_Max_Gust_kmh",
        "Monthly_Mean_Gust_Direction_10sdeg",
    ]
]

float_cols = wind_monthly_1995_2025.select_dtypes(include="float").columns
wind_monthly_1995_2025[float_cols] = wind_monthly_1995_2025[float_cols].round(2)
wind_monthly_1995_2025[float_cols] = wind_monthly_1995_2025[float_cols].astype("float32")


In [104]:
wind_monthly_1995_2025.head()

,Climate ID,Date,Monthly_Max_Gust_kmh,Monthly_Mean_Gust_Direction_10sdeg
0,8100467,2007-08-01,91.0,22.299999
1,8100467,2007-09-01,72.0,25.520000
2,8100467,2007-10-01,69.0,24.160000
3,8100467,2007-11-01,93.0,20.240000
4,8100467,2007-12-01,96.0,23.030001


In [105]:
wind_monthly_1995_2025.info()

<class 'pandas.DataFrame'>
RangeIndex: 21309 entries, 0 to 21308
Data columns (total 4 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Climate ID                          21309 non-null  str           
 1   Date                                21309 non-null  datetime64[us]
 2   Monthly_Max_Gust_kmh                21309 non-null  float32       
 3   Monthly_Mean_Gust_Direction_10sdeg  21132 non-null  float32       
dtypes: datetime64[us](1), float32(2), str(1)
memory usage: 647.8 KB


In [106]:
wind_monthly_1995_2025.to_parquet(
    r"../../../../data/clean/Yashaswi/Final_Clean/monthly_wind_1995_2025.parquet",
    index=False
)